## Imports

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq
from scipy.ndimage import gaussian_filter1d

## File paths

In [2]:
CSEG_DIR = Path('/glade/campaign/cesm/cesmdata/cseg/inputdata/atm/cam/scam/iop')
NCDATA_DIR = Path('/glade/work/rneale/scam_cases/ncdata')
OUT_DIR  = NCDATA_DIR

# Source files (L32)
SRC_FILES = [
    CSEG_DIR / 'CESM2.F2000climo.64x128.cam.i.0003-01-01-00000.nc',
    CSEG_DIR / 'CESM2.F2000climo.64x128.cam.i.0003-07-01-00000.nc',
]

# Reference files for target grids
L48_REF = NCDATA_DIR / 'FWsc_T42_48L_GRID_48_taperstart10km_lowtop_Top_42km.nc'
L58_REF = NCDATA_DIR / 'FWsc_T42_58L_GRID_48_taperstart10km_lowtop_BL10_v3_beta1p75_Top_43km.nc'

# L256 grid construction method:
#   'cubic_spline' – original: cubic spline of L32 interface coeffs at uniform η
#   'smooth_sfc'   – exponential height spacing with 15 m bottom layer + near-surface smoothing
L256_METHOD = 'smooth_sfc'

for f in SRC_FILES:
    print('Source:', f.name)
print('L48 ref:', L48_REF.name)
print('L58 ref:', L58_REF.name)
print('L256 method:', L256_METHOD)

Source: CESM2.F2000climo.64x128.cam.i.0003-01-01-00000.nc
Source: CESM2.F2000climo.64x128.cam.i.0003-07-01-00000.nc
L48 ref: FWsc_T42_48L_GRID_48_taperstart10km_lowtop_Top_42km.nc
L58 ref: FWsc_T42_58L_GRID_48_taperstart10km_lowtop_BL10_v3_beta1p75_Top_43km.nc
L256 method: smooth_sfc


# Helper functions

In [7]:
def hybrid_pressure(hyam, hybm, P0, PS):
    """Pressure at each model level: (lev, lat, lon)."""
    return (hyam[:, np.newaxis, np.newaxis] * P0
            + hybm[:, np.newaxis, np.newaxis] * PS[np.newaxis, :, :])


def vinterp_column(src_vals, log_src_p, log_tgt_p):
    """Linear interpolation in log-pressure for one column.
    Boundary values are held constant outside the source range.
    """
    idx = np.searchsorted(log_src_p, log_tgt_p).clip(1, len(log_src_p) - 1)
    p0 = log_src_p[idx - 1];  p1 = log_src_p[idx]
    v0 = src_vals[idx - 1];   v1 = src_vals[idx]
    dp = p1 - p0
    wt = np.where(dp != 0.0, (log_tgt_p - p0) / dp, 0.5).clip(0.0, 1.0)
    out = v0 + wt * (v1 - v0)
    out[log_tgt_p <= log_src_p[0]]  = src_vals[0]
    out[log_tgt_p >= log_src_p[-1]] = src_vals[-1]
    return out


def vinterp_field(data, src_pres, tgt_pres):
    """Interpolate (lev_src, lat, lon) → (lev_tgt, lat, lon)."""
    nlev_tgt     = tgt_pres.shape[0]
    lat_n, lon_n = data.shape[1], data.shape[2]
    n_col        = lat_n * lon_n
    d_flat   = data.reshape(-1, n_col)
    log_sp   = np.log(src_pres.reshape(-1, n_col))
    log_tp   = np.log(tgt_pres.reshape(-1, n_col))
    out_flat = np.empty((nlev_tgt, n_col), dtype=data.dtype)
    for j in range(n_col):
        out_flat[:, j] = vinterp_column(d_flat[:, j], log_sp[:, j], log_tp[:, j])
    return out_flat.reshape(nlev_tgt, lat_n, lon_n)


def make_L256_grid(hyai_src, hybi_src):
    """Build a smooth 256-level hybrid grid from source interface coefficients.
    A cubic spline through the source interfaces is resampled at 257 evenly
    spaced points, giving 256 layers with no abrupt thickness jumps.
    """
    n_src = len(hyai_src)
    n_ifc = 257              # → 256 layers
    eta_src = np.linspace(0.0, 1.0, n_src)
    eta_tgt = np.linspace(0.0, 1.0, n_ifc)
    hyai_new = np.clip(CubicSpline(eta_src, hyai_src)(eta_tgt), 0.0, None)
    hybi_new = np.clip(CubicSpline(eta_src, hybi_src)(eta_tgt), 0.0, None)
    # Enforce boundary conditions
    hyai_new[0]  = hyai_src[0];  hybi_new[0]  = 0.0   # top: pure pressure
    hyai_new[-1] = 0.0;           hybi_new[-1] = 1.0   # bottom: pure sigma
    hyam_new = 0.5 * (hyai_new[:-1] + hyai_new[1:])
    hybm_new = 0.5 * (hybi_new[:-1] + hybi_new[1:])
    lev_new  = 1000.0 * (hyam_new + hybm_new)
    ilev_new = 1000.0 * (hyai_new + hybi_new)
    return dict(hyai=hyai_new, hybi=hybi_new,
                hyam=hyam_new, hybm=hybm_new,
                lev=lev_new,   ilev=ilev_new,
                n_lev=256)


def make_L256_grid_smooth_sfc(hyai_src, hybi_src, P0=101325.,
                               dz_bot=15., H_scale=7000.,
                               n_sm=30, sigma_sm=2.5):
    """Build a 256-level hybrid grid with 15 m bottom layer and smooth near-surface spacing.

    The same cubic spline as make_L256_grid is used, but instead of sampling
    it at uniform η points, sample points are derived from exponentially-spaced
    interface heights (bottom thickness = dz_bot).  A light Gaussian smoothing
    is then applied to the lowest n_sm layer thicknesses so the profile near the
    surface ramps up without any kinks.

    Parameters
    ----------
    hyai_src, hybi_src : source (e.g. L32) interface hybrid coefficients
    P0       : reference pressure (Pa)
    dz_bot   : target bottom-layer thickness (m)
    H_scale  : isothermal scale height for z↔p conversion (m)
    n_sm     : number of bottom layers to Gaussian-smooth
    sigma_sm : Gaussian sigma (layer units) for near-surface smoothing
    """
    n_src = len(hyai_src)
    n_ifc = 257   # → 256 layers
    N     = 256

    PS_ref  = P0
    p_top   = hyai_src[0]  * P0 + hybi_src[0]  * PS_ref   # Pa
    p_sfc   = hyai_src[-1] * P0 + hybi_src[-1] * PS_ref   # Pa  (≈ P0)
    z_top   = H_scale * np.log(p_sfc / p_top)

    # ── Exponential interface heights: z_i = C*(exp(alpha*i/N) - 1) ──
    # z_int runs surface→top: z_int[0]=0 (sfc), z_int[-1]=z_top
    def alpha_eq(alpha):
        C = z_top / (np.exp(alpha) - 1.)
        return C * (np.exp(alpha / N) - 1.) - dz_bot

    alpha_sol = brentq(alpha_eq, 0.1, 30.)
    C_sol = z_top / (np.exp(alpha_sol) - 1.)

    i_arr = np.arange(n_ifc)
    z_int = C_sol * (np.exp(alpha_sol * i_arr / N) - 1.)
    dz    = np.diff(z_int)

    # ── Light Gaussian smoothing on lowest n_sm layers ──
    dz_sm = dz.copy()
    dz_sm[:n_sm] = gaussian_filter1d(dz[:n_sm], sigma=sigma_sm)

    # Rebuild interface heights from smoothed dz; rescale to preserve z_top
    z_int_sm = np.zeros(n_ifc)
    for ii in range(N):
        z_int_sm[ii + 1] = z_int_sm[ii] + dz_sm[ii]
    z_int_sm *= z_top / z_int_sm[-1]

    print(f'  L256 smooth_sfc: bottom dz = {np.diff(z_int_sm)[0]:.1f} m  '
          f'(target {dz_bot} m),  alpha = {alpha_sol:.3f}')

    # ── Convert heights → pressure → η, then sample the spline ──
    # z_int_sm runs surface→top; flip to CAM convention (top→surface) so that
    # η increases from 0 (top) to 1 (surface), matching the spline ordering.
    p_int_sm = p_sfc * np.exp(-z_int_sm[::-1] / H_scale)
    eta_tgt  = np.clip((p_int_sm - p_top) / (p_sfc - p_top), 0., 1.)

    eta_src  = np.linspace(0., 1., n_src)
    cs_a     = CubicSpline(eta_src, hyai_src)
    cs_b     = CubicSpline(eta_src, hybi_src)

    hyai_new = np.clip(cs_a(eta_tgt), 0., None)
    hybi_new = np.clip(cs_b(eta_tgt), 0., None)

    # Enforce boundary conditions
    hyai_new[0]  = hyai_src[0];  hybi_new[0]  = 0.0   # top: pure pressure
    hyai_new[-1] = 0.0;           hybi_new[-1] = 1.0   # bottom: pure sigma

    hyam_new = 0.5 * (hyai_new[:-1] + hyai_new[1:])
    hybm_new = 0.5 * (hybi_new[:-1] + hybi_new[1:])
    lev_new  = 1000. * (hyam_new + hybm_new)
    ilev_new = 1000. * (hyai_new + hybi_new)

    return dict(hyai=hyai_new, hybi=hybi_new,
                hyam=hyam_new, hybm=hybm_new,
                lev=lev_new,   ilev=ilev_new,
                n_lev=256)


# Vertical coordinate variables replaced separately — exclude from interpolation loop
_COORD_VARS = {'lev', 'ilev', 'hyam', 'hybm', 'hyai', 'hybi'}


def write_interpolated(src_ds, tgt_grid, out_path):
    """Interpolate all lev-dependent 4-D variables to tgt_grid and write netCDF."""
    P0       = float(src_ds['P0'])
    PS       = src_ds['PS'].values
    hyam_src = src_ds['hyam'].values
    hybm_src = src_ds['hybm'].values
    hyam_tgt = tgt_grid['hyam']
    hybm_tgt = tgt_grid['hybm']
    n_lev    = tgt_grid['n_lev']
    n_time   = PS.shape[0]

    print(f'  → {out_path.name}  ({n_lev} levels)')
    ds_out = xr.Dataset()

    # Copy non-level variables unchanged
    for vname, var in src_ds.items():
        if 'lev' not in var.dims and 'ilev' not in var.dims:
            ds_out[vname] = var

    # Replace vertical coordinate variables
    ds_out['lev']  = xr.DataArray(tgt_grid['lev'],  dims='lev',  attrs=src_ds['lev'].attrs)
    ds_out['ilev'] = xr.DataArray(tgt_grid['ilev'], dims='ilev', attrs=src_ds['ilev'].attrs)
    ds_out['hyam'] = xr.DataArray(tgt_grid['hyam'], dims='lev',  attrs=src_ds['hyam'].attrs)
    ds_out['hybm'] = xr.DataArray(tgt_grid['hybm'], dims='lev',  attrs=src_ds['hybm'].attrs)
    ds_out['hyai'] = xr.DataArray(tgt_grid['hyai'], dims='ilev', attrs=src_ds['hyai'].attrs)
    ds_out['hybi'] = xr.DataArray(tgt_grid['hybi'], dims='ilev', attrs=src_ds['hybi'].attrs)

    # Interpolate 4-D lev-dependent variables (time, lev, lat, lon)
    lev_vars = [v for v, da in src_ds.items()
                if 'lev' in da.dims and v not in _COORD_VARS and da.ndim == 4]
    for vname in lev_vars:
        src_var = src_ds[vname].values
        lat_n, lon_n = src_var.shape[2], src_var.shape[3]
        out_arr = np.empty((n_time, n_lev, lat_n, lon_n), dtype=src_var.dtype)
        print(f'     interpolating {vname} ...', end='', flush=True)
        for it in range(n_time):
            src_p = hybrid_pressure(hyam_src, hybm_src, P0, PS[it])
            tgt_p = hybrid_pressure(hyam_tgt, hybm_tgt, P0, PS[it])
            out_arr[it] = vinterp_field(src_var[it], src_p, tgt_p)
        if vname == 'Q':
            out_arr = np.clip(out_arr, 0.0, None)
        ds_out[vname] = xr.DataArray(out_arr, dims=src_ds[vname].dims, attrs=src_ds[vname].attrs)
        print(' done')

    for cname in ('lat', 'lon', 'time'):
        if cname in src_ds.coords:
            ds_out = ds_out.assign_coords({cname: src_ds[cname]})
    ds_out.attrs = dict(src_ds.attrs)
    ds_out.attrs['history'] = (
        f'Vertically interpolated to {n_lev} hybrid levels by vinterp_cesm2_climo.ipynb\n'
        + src_ds.attrs.get('history', '')
    )
    # Remove any existing file first — NetCDF4 can't overwrite a file that is
    # still held open in the kernel from a previous run.
    out_path.unlink(missing_ok=True)
    ds_out.to_netcdf(out_path)
    print(f'  Saved: {out_path}')

## Load reference grids

In [8]:
L48_ds = xr.open_dataset(L48_REF)
L58_ds = xr.open_dataset(L58_REF)

L48_grid = dict(
    hyai  = L48_ds['hyai'].values,
    hybi  = L48_ds['hybi'].values,
    hyam  = L48_ds['hyam'].values,
    hybm  = L48_ds['hybm'].values,
    lev   = L48_ds['lev'].values,
    ilev  = L48_ds['ilev'].values,
    n_lev = int(L48_ds.sizes['lev']),
)
L58_grid = dict(
    hyai  = L58_ds['hyai'].values,
    hybi  = L58_ds['hybi'].values,
    hyam  = L58_ds['hyam'].values,
    hybm  = L58_ds['hybm'].values,
    lev   = L58_ds['lev'].values,
    ilev  = L58_ds['ilev'].values,
    n_lev = int(L58_ds.sizes['lev']),
)

print(f'L48 grid: {L48_grid["n_lev"]} levels')
print(f'L58 grid: {L58_grid["n_lev"]} levels')

L48 grid: 48 levels
L58 grid: 58 levels


## Interpolate both source files

In [9]:
for src_file in SRC_FILES:
    print(f'\n=== {src_file.name} ===')
    src_ds = xr.open_dataset(src_file)
    print(f'  Source levels : {src_ds.sizes["lev"]}')
    print(f'  Horizontal    : {src_ds.sizes["lat"]} lat × {src_ds.sizes["lon"]} lon')
    print(f'  Time steps    : {src_ds.sizes["time"]}')

    stem = src_file.stem   # e.g. CESM2.F2000climo.64x128.cam.i.0003-01-01-00000

    # L48
    write_interpolated(src_ds, L48_grid,  OUT_DIR / f'{stem}_L48.nc')

    # L58
    write_interpolated(src_ds, L58_grid,  OUT_DIR / f'{stem}_L58.nc')

    # L256 — grid construction method selected by L256_METHOD
    hyai_src = src_ds['hyai'].values
    hybi_src = src_ds['hybi'].values
    if L256_METHOD == 'smooth_sfc':
        print('  Building L256 grid: smooth_sfc (exponential spacing, 15 m bottom layer) ...')
        L256_grid = make_L256_grid_smooth_sfc(hyai_src, hybi_src)
    else:  # 'cubic_spline'
        print('  Building L256 grid: cubic_spline (uniform η resampling) ...')
        L256_grid = make_L256_grid(hyai_src, hybi_src)
    write_interpolated(src_ds, L256_grid, OUT_DIR / f'{stem}_L256.nc')

    src_ds.close()

print('\nAll done.')


=== CESM2.F2000climo.64x128.cam.i.0003-01-01-00000.nc ===
  Source levels : 32
  Horizontal    : 64 lat × 128 lon
  Time steps    : 1
  → CESM2.F2000climo.64x128.cam.i.0003-01-01-00000_L48.nc  (48 levels)
     interpolating CLDICE ... done
     interpolating CLDLIQ ... done
     interpolating DMS ... done
     interpolating H2O2 ... done
     interpolating H2SO4 ... done
     interpolating NUMICE ... done
     interpolating NUMLIQ ... done
     interpolating NUMRAI ... done
     interpolating NUMSNO ... done
     interpolating Q ... done
     interpolating RAINQM ... done
     interpolating SNOWQM ... done
     interpolating SO2 ... done
     interpolating SOAG ... done
     interpolating T ... done
     interpolating U ... done
     interpolating V ... done
     interpolating bc_a1 ... done
     interpolating bc_a4 ... done
     interpolating dst_a1 ... done
     interpolating dst_a2 ... done
     interpolating dst_a3 ... done
     interpolating ncl_a1 ... done
     interpolating ncl